# DSA211 Topic 2 — dataset selection

## How we chose the data

We began with two candidates, ACRA Singapore firm survival and Taiwan credit default, then added four more to test whether either was genuinely good or merely first to hand. A candidate had to be public, binary-outcome, not course data, and preferably Asian; it also had to carry enough positives to survive a held-out split plus 5-fold CV, and to mix continuous with categorical features, so that LDA's and Naive Bayes' distributional assumptions are live questions rather than formalities. Bank Marketing (Portugal) was included deliberately as a **non-Asian control**: it is the closest public match to this brief's requirements, so it tells us whether the Asian shortlist is actually competitive or just the best of a weak field.

## How we tested each one

Every dataset runs through one identical pipeline. The held-out set is split off first and scored exactly once at the end; all imputation, scaling and one-hot encoding sit inside the scikit-learn `Pipeline`, so they are refitted on each training fold rather than on the full training set; declared leakage suspects are priced (naive AUC minus audited AUC) and then dropped; and every operating threshold is chosen by minimising expected cost on **training out-of-fold predictions only**. We then score each dataset against fifteen criteria reverse-engineered from the assignment, where each criterion asks whether a required section of the report can actually be written — for example, whether the three models yield three *distinct* confusion matrices, and whether the cost argument can actually pick a model. This step matters: an earlier nine-criterion proxy passed ACRA 9 of 9, and running the real requirements revealed its deliverable was degenerate.

## Which dataset to use

**Use Online Shoppers Purchasing Intention (UCI 468).** It is the only one of the six to pass all fifteen criteria, because it is the only dataset that simultaneously supports a real leakage audit, a genuine calibration contrast across the three models, and a cost argument that actually discriminates between them. If the Asian requirement must be read strictly — Turkey is transcontinental — then use Taiwan credit default, which passes 14 of 15.

---
*Everything below reproduces the result. Run top to bottom; downloads are cached in `.cache/`.*

## 1. Imports and data

In [ ]:
import io, os, time, warnings
import numpy as np, pandas as pd, requests
import matplotlib; import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.model_selection import StratifiedKFold, cross_val_predict, train_test_split
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix, brier_score_loss
warnings.filterwarnings("ignore")
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 60)

SEED, TEST_SIZE, K, COST = 42, 0.25, 5, 5.0    # COST = C_FN / C_FP, so t* = 1/(1+5) = 0.167
CACHE = ".cache"; os.makedirs(CACHE, exist_ok=True)
RUN_ACRA = True                                 # first run downloads ~170 MB of ACRA letter files

MODELS = {"Logistic":   lambda: LogisticRegression(max_iter=3000),
          "GaussianNB": lambda: GaussianNB(),
          "LDA":        lambda: LDA(solver="svd")}
print(f"cost ratio {COST} -> theoretical threshold {1/(1+COST):.4f}")

In [ ]:
def _get(fname, url):
    """Download once, cache on disk. requests carries certifi; pandas' urllib does not."""
    p = os.path.join(CACHE, fname)
    if not (os.path.exists(p) and os.path.getsize(p) > 500):
        r = requests.get(url, timeout=900); r.raise_for_status()
        open(p, "wb").write(r.content)
    return p

# ucimlrepo's CSV endpoint currently raises DatasetNotFoundError for every id
# tested, so we read UCI's static data_url directly.
def uci(i):
    return pd.read_csv(_get(f"uci_{i}.csv",
        f"https://archive.ics.uci.edu/static/public/{i}/data.csv"), low_memory=False)

# --- adapters: each returns (X, y, leakage_suspects, origin) ----------------
# A "suspect" is declared BEFORE modelling, measured via the leakage premium,
# then dropped. Declaring one is a judgement call, so each carries its reason.

TW = ["LIMIT_BAL","SEX","EDUCATION","MARRIAGE","AGE","PAY_0","PAY_2","PAY_3","PAY_4","PAY_5",
      "PAY_6","BILL_AMT1","BILL_AMT2","BILL_AMT3","BILL_AMT4","BILL_AMT5","BILL_AMT6",
      "PAY_AMT1","PAY_AMT2","PAY_AMT3","PAY_AMT4","PAY_AMT5","PAY_AMT6"]

def d_taiwan():
    d = uci(350).drop(columns=["ID"]).rename(columns={f"X{i+1}": n for i, n in enumerate(TW)})
    return d.drop(columns=["Y"]), d["Y"].astype(int), [], "Taiwan (Asia)"

def d_shoppers():
    d = uci(468); y = d.pop("Revenue").astype(bool).astype(int)
    for c in ["OperatingSystems","Browser","Region","TrafficType"]:
        d[c] = d[c].astype(str)          # integer-coded nominal IDs, not quantities
    # PageValues is a Google Analytics figure averaged over sessions that converted,
    # so it is partly derived from the outcome.
    return d, y, ["PageValues"], "Turkey (Asia, transcontinental)"

def d_cmc():
    d = uci(30); y = (d.pop("contraceptive_method") != 1).astype(int)   # 3-class -> use vs no-use
    d["husband_occupation"] = d["husband_occupation"].astype(str)
    return d, y, [], "Indonesia (Asia)"

def d_bank():
    d = uci(222); y = (d.pop("y").astype(str).str.strip().str.lower() == "yes").astype(int)
    # 'duration' is the length of the call being predicted; unknown until the
    # outcome exists. UCI's own page says to discard it for realistic models.
    return d, y, ["duration"], "Portugal (CONTROL, not Asian)"

def d_twbank():
    d = uci(572); d.columns = [c.strip() for c in d.columns]
    return d.drop(columns=["Bankrupt?"]), d["Bankrupt?"].astype(int), [], "Taiwan (Asia)"

ACRA_IDS = {"A":"d_8575e84912df3c28995b8e6e0e05205a","G":"d_b58303c68e9cf0d2ae93b73ffdbfbfa1",
            "S":"d_df7d2d661c0c11a7c367c9ee4bf896c1","X":"d_1cd970d8351b42be4a308d628a6dd9d3",
            "Z":"d_4e3db8955fdcda6f9944097bef3d2724"}
ACRA_COLS = ["entity_status_description","registration_incorporation_date","entity_type_description",
             "company_type_description","business_constitution_description","primary_ssic_code",
             "postal_code","no_of_officers","account_due_date","annual_return_date"]

def d_acra(cohort=(2010, 2015)):
    fr = []
    for L, i in ACRA_IDS.items():
        p = os.path.join(CACHE, f"acra_{L}.csv")
        if not os.path.exists(p):
            u = requests.get("https://api-open.data.gov.sg/v1/public/api/datasets/"
                             f"{i}/poll-download", timeout=120).json()["data"]["url"]
            _get(f"acra_{L}.csv", u); time.sleep(8)      # the API rate-limits at 429
        fr.append(pd.read_csv(p, usecols=ACRA_COLS, dtype=str, low_memory=False))
    d = pd.concat(fr, ignore_index=True)
    inc = pd.to_datetime(d["registration_incorporation_date"], errors="coerce", format="mixed")
    d["inc_year"], d["inc_month"] = inc.dt.year, inc.dt.month
    d = d[d["inc_year"].between(*cohort)].copy()         # equalise exposure time
    s = d["entity_status_description"].str.lower().str.strip()
    pos, neg = s.str.contains("struck", na=False), s.str.startswith("live", na=False)
    d, y = d[pos | neg].copy(), pos[pos | neg].astype(int)
    d["ssic2"]   = d["primary_ssic_code"].astype(str).str.extract(r"^(\d{2})")[0]
    d["postal2"] = d["postal_code"].astype(str).str.extract(r"^(\d{2})")[0]
    # suspects: a struck-off company has no officers and stale filings
    d["sus_officers"] = pd.to_numeric(d["no_of_officers"], errors="coerce")
    X = d[["entity_type_description","company_type_description","business_constitution_description",
           "ssic2","postal2","inc_year","inc_month","sus_officers"]]
    return X, y, ["sus_officers"], "Singapore (Asia)"

LOADERS = {"Taiwan credit default": d_taiwan, "Online shoppers intent": d_shoppers,
           "Contraceptive choice": d_cmc,     "Bank marketing": d_bank,
           "Taiwan bankruptcy": d_twbank}
if RUN_ACRA: LOADERS["ACRA firm survival"] = d_acra

DATA = {}
for n, f in LOADERS.items():
    X, y, susp, origin = f()
    DATA[n] = dict(X=X, y=y, suspects=susp, origin=origin)
    print(f"{n:<26} rows {len(X):>7,}  pos {int(y.sum()):>6,} ({y.mean():.3f})  "
          f"cols {X.shape[1]:>3}  suspects {susp or 'none'}")

## 2. Basic EDA

In [ ]:
def split_types(df, max_card=12):
    """Numeric if continuous; a numeric column with few levels is really categorical."""
    num, cat = [], []
    for c in df.columns:
        s = df[c]
        if pd.api.types.is_numeric_dtype(s):
            (cat if s.nunique(dropna=True) <= max_card else num).append(c)
        else:
            cat.append(c)
    return num, cat

eda = []
for n, d in DATA.items():
    X, y = d["X"], d["y"]
    num, cat = split_types(X)
    eda.append({"dataset": n, "origin": d["origin"], "rows": len(X), "cols": X.shape[1],
                "positives": int(y.sum()), "prevalence": round(float(y.mean()), 4),
                "numeric": len(num), "categorical": len(cat),
                "pct missing": round(float(X.isna().to_numpy().mean()), 4),
                "suspects": ", ".join(d["suspects"]) or "none"})
display(pd.DataFrame(eda).set_index("dataset"))

In [ ]:
fig, ax = plt.subplots(1, len(DATA), figsize=(3.0*len(DATA), 2.6), sharey=True)
for a, (n, d) in zip(np.atleast_1d(ax), DATA.items()):
    v = d["y"].value_counts().sort_index()
    a.bar(["neg","pos"], v.values, color=["#9fb3c8","#c0562c"])
    a.set_title(f"{n}\nprev {d['y'].mean():.3f}", fontsize=8)
    a.tick_params(labelsize=7)
np.atleast_1d(ax)[0].set_ylabel("rows")
plt.tight_layout(); plt.show()

## 3. Evaluation engine

One pipeline for every dataset. Split first, preprocess inside the fold, threshold from training out-of-fold predictions, test set touched once.

In [ ]:
def make_pre(num, cat):
    """All preprocessing lives here, so ColumnTransformer is refitted per fold."""
    parts = []
    if num: parts.append(("num", Pipeline([("i", SimpleImputer(strategy="median")),
                                           ("s", StandardScaler())]), num))
    if cat: parts.append(("cat", Pipeline([("i", SimpleImputer(strategy="most_frequent")),
                                           ("o", OneHotEncoder(drop="if_binary",
                                                handle_unknown="infrequent_if_exist",
                                                min_frequency=0.005, max_categories=30,
                                                sparse_output=False))]), cat))
    return ColumnTransformer(parts, remainder="drop")

def cost_curve(y, p, ratio=COST):
    ts = np.linspace(0.005, 0.995, 199); y = np.asarray(y)
    P, N = (y == 1).sum(), (y == 0).sum()
    cost, sens, spec = [], [], []
    for t in ts:
        pr = (p >= t).astype(int)
        tp = ((pr == 1) & (y == 1)).sum(); fp = ((pr == 1) & (y == 0)).sum()
        cost.append(((P - tp) * ratio + fp) / len(y)); sens.append(tp / P); spec.append((N - fp) / N)
    return ts, np.array(cost), np.array(sens), np.array(spec)

def cm(y, p, t):
    tn, fp, fn, tp = confusion_matrix(y, (np.asarray(p) >= t).astype(int), labels=[0, 1]).ravel()
    return dict(TN=int(tn), FP=int(fp), FN=int(fn), TP=int(tp),
                sens=tp/(tp+fn) if tp+fn else np.nan, spec=tn/(tn+fp) if tn+fp else np.nan)

def wilson(k, n):
    if not n: return (np.nan, np.nan)
    z = 1.959963985; p = k/n; d = 1 + z*z/n
    c = (p + z*z/(2*n))/d; h = z*np.sqrt(p*(1-p)/n + z*z/(4*n*n))/d
    return max(0.0, c-h), min(1.0, c+h)

In [ ]:
def evaluate(name, X, y, suspects, origin):
    X = X.reset_index(drop=True)
    y = pd.Series(np.asarray(y)).reset_index(drop=True).astype(int)

    # (1) held-out split happens FIRST and is scored once, at the end
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=TEST_SIZE, stratify=y, random_state=SEED)
    Xtr, Xte = Xtr.reset_index(drop=True), Xte.reset_index(drop=True)
    ytr, yte = ytr.reset_index(drop=True), yte.reset_index(drop=True)

    keep = [c for c in Xtr.columns if c not in set(suspects)]     # (2) drop declared suspects
    Xa, Xb = Xtr[keep], Xte[keep]
    num, cat = split_types(Xa)
    skf = StratifiedKFold(K, shuffle=True, random_state=SEED)

    def cvp(frame, mk):                                           # (3) preprocessing inside the fold
        pipe = Pipeline([("pre", make_pre(*split_types(frame))), ("clf", mk())])
        return cross_val_predict(pipe, frame, ytr, cv=skf, method="predict_proba")[:, 1]

    lr = lambda: LogisticRegression(max_iter=3000)
    premium = (roc_auc_score(ytr, cvp(Xtr, lr)) - roc_auc_score(ytr, cvp(Xa, lr))) if suspects else 0.0

    rows, oof = {}, {}
    for m, mk in MODELS.items():
        p = cvp(Xa, mk); oof[m] = p
        ts, cost, _, _ = cost_curve(ytr, p)
        t = float(ts[cost.argmin()])                              # (4) threshold from TRAIN OOF only
        fitted = Pipeline([("pre", make_pre(num, cat)), ("clf", mk())]).fit(Xa, ytr)
        pte = fitted.predict_proba(Xb)[:, 1]                      # (5) the one look at the test set
        st  = cm(yte, pte, t); lo, hi = wilson(st["TP"], st["TP"] + st["FN"])
        rows[m] = dict(cv_auc=roc_auc_score(ytr, p), brier=brier_score_loss(ytr, p), t=t,
                       cost_t=float(cost.min()), cost_half=float(cost[np.abs(ts-0.5).argmin()]),
                       test_auc=roc_auc_score(yte, pte), **st, ci_w=hi-lo,
                       folds=np.array([roc_auc_score(ytr.iloc[v], p[v]) for _, v in skf.split(Xa, ytr)]))

    R = pd.DataFrame({m: {k: v for k, v in r.items() if k != "folds"} for m, r in rows.items()}).T
    best, worst = R["cv_auc"].idxmax(), R["cv_auc"].idxmin()
    d  = rows[best]["folds"] - rows[worst]["folds"]; se = d.std(ddof=1)/np.sqrt(len(d))
    cs = sorted(R["cost_t"]); prev = float(ytr.mean()); triv = min(prev*COST, 1-prev)
    o  = R.loc[best]
    S = dict(dataset=name, origin=origin, rows=len(X), prevalence=float(y.mean()),
             n_train_pos=int(ytr.sum()), n_test_pos=int(yte.sum()),
             n_numeric=len(num), n_categorical=len(cat), leak_premium=float(premium),
             distinct_cm=len({(r["TN"], r["FP"], r["FN"], r["TP"]) for r in rows.values()}),
             separated=bool(abs(d.mean()) > 2*se),
             best_auc=float(R["cv_auc"].max()),
             brier_spread=float(R["brier"].max() - R["brier"].min()),
             cost_gap=float((cs[1]-cs[0])/cs[0]) if cs[0] > 0 else np.nan,
             cut_half=float((o["cost_half"]-o["cost_t"])/o["cost_half"]),
             cut_triv=float((triv-o["cost_t"])/triv),
             test_sens=float(o["sens"]), test_spec=float(o["spec"]),
             min_ss=float(min(o["sens"], o["spec"])),
             cv_test_gap=abs(float(o["cv_auc"]-o["test_auc"])), ci_w=float(o["ci_w"]),
             best_model=best, best_by_cost=str(R["cost_t"].idxmin()))
    return S, R, oof, ytr

RES = {}
for n, d in DATA.items():
    RES[n] = evaluate(n, d["X"], d["y"], d["suspects"], d["origin"])
    print(f"done: {n}")

## 4. Results

In [ ]:
cols = ["cv_auc","brier","t","cost_t","test_auc","sens","spec","TN","FP","FN","TP"]
for n, (S, R, _, _) in RES.items():
    print(f"\n=== {n}  [{S['origin']}]   leakage premium {S['leak_premium']:.4f}")
    print(R[cols].astype(float).round(4).to_string())

In [ ]:
fig, ax = plt.subplots(1, len(RES), figsize=(3.1*len(RES), 3.0))
for a, (n, (S, R, oof, ytr)) in zip(np.atleast_1d(ax), RES.items()):
    for m in MODELS:
        fpr, tpr, _ = roc_curve(ytr, oof[m])
        a.plot(fpr, tpr, lw=1.4, label=f"{m} {R.loc[m,'cv_auc']:.3f}")
    a.plot([0,1],[0,1],"k--",lw=.7)
    a.set_title(n, fontsize=8); a.set_xlabel("1 - specificity", fontsize=7)
    a.legend(fontsize=6, loc="lower right"); a.tick_params(labelsize=6)
np.atleast_1d(ax)[0].set_ylabel("sensitivity", fontsize=7)
plt.tight_layout(); plt.show()

In [ ]:
# 15 criteria reverse-engineered from the brief. Each asks whether a required
# section of the report can actually be written.
CRIT = [
 ("Train positives >= 500",        "n_train_pos",  lambda v: v >= 500,          "k-fold CV section"),
 ("Held-out positives >= 100",     "n_test_pos",   lambda v: v >= 100,          "CV vs held-out"),
 ("Numeric features >= 3",         "n_numeric",    lambda v: v >= 3,            "scaling + LDA/NB"),
 ("Categorical features >= 3",     "n_categorical",lambda v: v >= 3,            "per-fold encoding"),
 ("3 distinct confusion matrices", "distinct_cm",  lambda v: v == 3,            "comparison table"),
 ("Models separate beyond noise",  "separated",    lambda v: bool(v),           "model comparison"),
 ("Operating point informative",   "min_ss",       lambda v: v >= 0.35,         "threshold extension"),
 ("Sensitivity not saturated",     "test_sens",    lambda v: v <= 0.98,         "threshold + CI"),
 ("Cost can pick a model",         "cost_gap",     lambda v: v >= 0.01,         "model justification"),
 ("Cost cut vs 0.5 >= 5%",         "cut_half",     lambda v: v >= 0.05,         "threshold extension"),
 ("Cost cut vs trivial >= 5%",     "cut_triv",     lambda v: v >= 0.05,         "threshold extension"),
 ("|CV - test AUC| <= 0.03",       "cv_test_gap",  lambda v: v <= 0.03,         "CV vs held-out"),
 ("Sens 95% CI width <= 0.20",     "ci_w",         lambda v: v <= 0.20,         "CV vs held-out"),
 ("Best CV AUC in [0.62, 0.95]",   "best_auc",     lambda v: 0.62 <= v <= 0.95, "whole report"),
 ("Brier spread >= 0.01",          "brier_spread", lambda v: v >= 0.01,         "calibration"),
]

rows, tally, fails = [], {}, {}
for label, key, test, breaks in CRIT:
    row = {"criterion": label}
    for n, (S, *_ ) in RES.items():
        v = S.get(key, np.nan)
        ok = False
        try: ok = bool(test(v)) and not (isinstance(v, float) and not np.isfinite(v))
        except Exception: ok = False
        tally[n] = tally.get(n, 0) + int(ok)
        if not ok: fails.setdefault(n, []).append((label, v, breaks))
        row[n] = f"{v:.4f} {'PASS' if ok else 'fail'}" if isinstance(v, (float, np.floating)) \
                 else f"{v} {'PASS' if ok else 'fail'}"
    rows.append(row)
SC = pd.DataFrame(rows).set_index("criterion")
display(SC)
display(pd.DataFrame(sorted(tally.items(), key=lambda kv: -kv[1]),
                     columns=["dataset", f"passed of {len(CRIT)}"]).set_index("dataset"))

## 5. Interpretation

In [ ]:
for n, t in sorted(tally.items(), key=lambda kv: -kv[1]):
    S = RES[n][0]
    print(f"\n{n}  [{S['origin']}]  {t}/{len(CRIT)}")
    print(f"   best model {S['best_model']} by AUC, {S['best_by_cost']} by cost"
          f"  |  leakage premium {S['leak_premium']:.4f}"
          f"  |  sens/spec {S['test_sens']:.3f}/{S['test_spec']:.3f}")
    for label, v, breaks in fails.get(n, []):
        vs = f"{v:.4f}" if isinstance(v, (float, np.floating)) else str(v)
        print(f"   FAIL {label} = {vs}  -> breaks {breaks}")
    if not fails.get(n): print("   no failures")

In [ ]:
# The two failure patterns worth naming, both detected rather than asserted.
print("A. Degenerate operating point (threshold extension collapses to a trivial rule):")
for n, (S, *_) in RES.items():
    if S["min_ss"] < 0.35:
        print(f"   {n:<24} sens {S['test_sens']:.3f} / spec {S['test_spec']:.3f}, "
              f"{S['n_numeric']} numeric features")

print("\nB. Cost cannot separate the top two models (so 'justify on cost' is empty):")
for n, (S, R, *_) in RES.items():
    if not (S["cost_gap"] >= 0.01):
        c = R["cost_t"].astype(float).sort_values()
        print(f"   {n:<24} gap {S['cost_gap']*100:6.3f}%   "
              + " | ".join(f"{i} {v:.6f}" for i, v in c.items()))

## 6. Conclusion

In [ ]:
n_crit = len(CRIT)
rank = sorted(tally.items(), key=lambda kv: -kv[1])
win, wn = rank[0]
asian = [(n, t) for n, t in rank if "CONTROL" not in RES[n][0]["origin"]]
ctrl  = [(n, t) for n, t in rank if "CONTROL" in RES[n][0]["origin"]]

print("RANKING")
for i, (n, t) in enumerate(rank, 1):
    S = RES[n][0]
    print(f"  {i}. {n:<24} {t:>2}/{n_crit}  {S['origin']:<32} "
          f"AUC {S['best_auc']:.4f}  sens/spec {S['test_sens']:.3f}/{S['test_spec']:.3f}")

print(f"\nWINNER: {win} ({wn}/{n_crit})")
if ctrl:
    cn, ct = ctrl[0]
    print(f"Control check: {cn} scored {ct}/{n_crit}. The Asian shortlist is "
          f"{'competitive with' if asian[0][1] >= ct else 'behind'} the non-Asian benchmark.")
ba, bt = asian[0]
if ba != win:
    print(f"Best strictly-Asian option: {ba} ({bt}/{n_crit}), costing {wn-bt} criteria.")
else:
    print(f"{win} also satisfies the Asian preference, so no trade is required.")

print(f"\nDECISION: use {win}. It passes {wn} of {n_crit}; "
      f"leakage premium {RES[win][0]['leak_premium']:.4f}, "
      f"Brier spread {RES[win][0]['brier_spread']:.4f}, "
      f"cost gap {RES[win][0]['cost_gap']*100:.2f}%.")